In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
import json
from pathlib import Path

import pandas as pd

from rich import print
from openai import OpenAI

In [3]:
client = OpenAI()

In [4]:
def download_job(id: str, file_path: str):
    batch = client.batches.retrieve(id)
    print(batch)

    if (batch.status == "completed"):
        file_content = client.files.content(batch.output_file_id).content
        with open(file_path, "wb") as f:
            f.write(file_content)

In [5]:
def gen_items(path: str):
    with (open(path, "r") as batch_file):
        for line in batch_file:
            parsed = json.loads(line)
            json_content = parsed["response"]["body"]["choices"][0]["message"]["content"]

            try:
                structured_output = json.loads(json_content)
                total_tokens = parsed["response"]["body"]["usage"]["total_tokens"]
                custom_id = parsed["custom_id"]

                yield {**{k.replace("_english", ""):v for k, v in structured_output.items()}, "custom_id": custom_id, "total_tokens": total_tokens}
            except:
                print("FAILED CUSTOM ID: " + parsed["custom_id"])

## Triplet

In [6]:
triplet_job_path = Path("../data/llm-gen/translated/triplet_batch_result.jsonl")
download_job("batch_681ed8f1a9d481909d0c19b6f0a8d969", triplet_job_path)

Batch(
    id='batch_681ed8f1a9d481909d0c19b6f0a8d969',
    completion_window='24h',
    created_at=1746852081,
    endpoint='/v1/chat/completions',
    input_file_id='file-Lfz56i7emPQTgaebnsT91N',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1746896324,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746938481,
    failed_at=None,
    finalizing_at=1746895148,
    in_progress_at=1746852085,
    metadata=None,
    output_file_id='file-TViMpzXaWwxLyocPfVMcdu',
    request_counts=BatchRequestCounts(completed=7959, failed=0, total=7959)
)

In [7]:
df_triplet = pd.read_json("../data/cleaned/triplet.jsonl", lines=True).reset_index()
df_triplet.head()

,index,query,positive,negative
0,0,Naon tujuan Holland Indonésé Festival?,Festival anu buka wakil duta besar walanda keu...,Urang walanda pohara resep ku masak urang indo...
1,1,Kapan Holland Indonésé Festival dilaksanakeun?,"Festival ieu dilaksanakeun di hotél salak, jal...",Sagalana rupa budaya milik bangsa indonésé dih...
2,2,Saha nu nyarios ngeunaan hubungan indonésé jeu...,"Wakil duta besar walanda, annemieke ruigrok ka...",Nurutkeun r. ay. suni wijogawati laku wakil pu...
3,3,Ala naon anu dipidangkeun di festival budaya ieu?,Sagalana rupa budaya milik bangsa indonésé dih...,Kuring gumbira aya di indonésé nu jalma saromé...
4,4,Saha anu mendakan cara sieun batik tulis?,Ruigrok sempet nengetan pengrajin batik mrakté...,Festival budaya ieu ngan gel sapoé.


In [8]:
df_triplet_map = pd.read_json("../data/llm-gen/translated/triplet_map.jsonl", lines=True)
df_triplet_map.head()

,custom_id,doc_index
0,0bb85ea3-2d73-4622-938b-2ff2124fd27f,0
1,59f8652c-a687-4d6c-aa3d-1b955ffdc6e3,1
2,b31446e2-f3cd-4086-aad0-903abad80576,2
3,837a6c31-fbfa-443b-84c4-64aebd48b580,3
4,9a438347-2f70-4f5a-bf9f-57edb8261e86,4


In [9]:
df_triplet_en = pd.DataFrame(gen_items(triplet_job_path))
df_triplet_en.head()

FAILED CUSTOM ID: b54387a5-fd07-4341-a3ab-37cdd52927b7

,query,positive,negative,custom_id,total_tokens
0,What is the purpose of the Holland Indonesian ...,"The festival, opened by the Dutch ambassador t...",The Dutch people really enjoy Indonesian cuisi...,0bb85ea3-2d73-4622-938b-2ff2124fd27f,288
1,When is the Holland Indonesian Festival held?,"This festival was held at Salak Hotel, Jl. Ir ...",Various cultural aspects of the Indonesian nat...,59f8652c-a687-4d6c-aa3d-1b955ffdc6e3,284
2,Who talks about the relationship between Indon...,"The Deputy Ambassador of the Netherlands, Anne...","According to R. Ay. Suni Wijogawati, a represe...",b31446e2-f3cd-4086-aad0-903abad80576,324
3,What is presented at this cultural festival?,All kinds of cultures belonging to the Indones...,I am happy to be in Indonesia where the people...,837a6c31-fbfa-443b-84c4-64aebd48b580,269
4,Who found the technique of fear in batik writing?,Ruigrok had the opportunity to observe batik a...,This cultural festival only lasts one day.,9a438347-2f70-4f5a-bf9f-57edb8261e86,243


In [10]:
print(f"Total tokens: {df_triplet_en['total_tokens'].sum()}")

Total tokens: 2113204

In [11]:
df_triplet_merged = df_triplet[["index"]].merge(df_triplet_map, left_on="index", right_on="doc_index")
df_triplet_merged = df_triplet_merged.merge(df_triplet_en, on="custom_id")
df_triplet_merged = df_triplet_merged.drop(columns=["index", "doc_index", "custom_id", "total_tokens"])
df_triplet_merged.head()

,query,positive,negative
0,What is the purpose of the Holland Indonesian ...,"The festival, opened by the Dutch ambassador t...",The Dutch people really enjoy Indonesian cuisi...
1,When is the Holland Indonesian Festival held?,"This festival was held at Salak Hotel, Jl. Ir ...",Various cultural aspects of the Indonesian nat...
2,Who talks about the relationship between Indon...,"The Deputy Ambassador of the Netherlands, Anne...","According to R. Ay. Suni Wijogawati, a represe..."
3,What is presented at this cultural festival?,All kinds of cultures belonging to the Indones...,I am happy to be in Indonesia where the people...
4,Who found the technique of fear in batik writing?,Ruigrok had the opportunity to observe batik a...,This cultural festival only lasts one day.


In [12]:
df_triplet_merged.to_json("../data/cleaned/triplet_en.jsonl", orient="records", lines=True)

## BEIR

### Queries

In [13]:
beir_queries_job_path = Path("../data/llm-gen/translated/beir_queries_batch_result.jsonl")
download_job("batch_681ed92d1b6881909495bbe442fba25b", beir_queries_job_path)

Batch(
    id='batch_681ed92d1b6881909495bbe442fba25b',
    completion_window='24h',
    created_at=1746852141,
    endpoint='/v1/chat/completions',
    input_file_id='file-RLAmgsk9bHmGv1uGHYSux5',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1746870791,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746938541,
    failed_at=None,
    finalizing_at=1746869618,
    in_progress_at=1746852206,
    metadata=None,
    output_file_id='file-5nn1MxD6ZxRgrWZDHxjjva',
    request_counts=BatchRequestCounts(completed=11612, failed=0, total=11612)
)

In [14]:
df_queries = pd.read_json("../data/cleaned/queries.jsonl", lines=True).reset_index()
df_queries.head()

,index,_id,text
0,0,99b78da2-54b4-4afa-b4c8-06df0dfb53d8,Holland Indonésé Festival di Bogor
1,1,42a37e80-e810-4813-a551-09dcbd471fa6,apa tujuan festival budaya di Bogor
2,2,5a0b9395-c28a-4c33-b5cc-5c502ee78790,siapa wakil duta besar Walanda
3,3,b821e85d-a6c3-4413-a0ba-d09ab1b72c2a,hubungan antara Indonésé jeung Walanda
4,4,37959229-85cc-432d-97e4-b437002efb17,acara naon anu dipidangkeun di festival


In [15]:
df_queries_map = pd.read_json("../data/llm-gen/translated/beir_query_map.jsonl", lines=True)
df_queries_map.head()

,custom_id,doc_index
0,3a189d35-fb8e-45c2-95cd-65d2c24156f1,0
1,321bd8ad-6ce5-430c-aa19-337fbd787ca7,1
2,a35703e7-9dc2-4c4f-9716-a600b24c8302,2
3,01c1365c-af5d-4983-b38f-c07c020c82e3,3
4,8e9100da-3b60-4bb0-9cca-9f064aaf87dd,4


In [16]:
df_queries_en = pd.DataFrame(gen_items(beir_queries_job_path))
df_queries_en.head()

,query,custom_id,total_tokens
0,Holland Indonesian Festival in Bogor,3a189d35-fb8e-45c2-95cd-65d2c24156f1,119
1,What is the purpose of the cultural festival i...,321bd8ad-6ce5-430c-aa19-337fbd787ca7,122
2,Who is the Deputy Ambassador of the Netherlands?,a35703e7-9dc2-4c4f-9716-a600b24c8302,121
3,the relationship between Indonesia and the Net...,01c1365c-af5d-4983-b38f-c07c020c82e3,119
4,What events are presented at the festival?,8e9100da-3b60-4bb0-9cca-9f064aaf87dd,121


In [17]:
print(f"Total tokens: {df_queries_en['total_tokens'].sum()}")

Total tokens: 1419931

In [18]:
df_queries_merged = df_queries[["index", "_id"]].merge(df_queries_map, left_on="index", right_on="doc_index")
df_queries_merged = df_queries_merged.merge(df_queries_en, on="custom_id")
df_queries_merged = df_queries_merged.drop(columns=["index", "doc_index", "custom_id", "total_tokens"])
df_queries_merged.head()

,_id,query
0,99b78da2-54b4-4afa-b4c8-06df0dfb53d8,Holland Indonesian Festival in Bogor
1,42a37e80-e810-4813-a551-09dcbd471fa6,What is the purpose of the cultural festival i...
2,5a0b9395-c28a-4c33-b5cc-5c502ee78790,Who is the Deputy Ambassador of the Netherlands?
3,b821e85d-a6c3-4413-a0ba-d09ab1b72c2a,the relationship between Indonesia and the Net...
4,37959229-85cc-432d-97e4-b437002efb17,What events are presented at the festival?


In [19]:
df_queries_merged.to_json("../data/cleaned/queries_en.jsonl", orient="records", lines=True)

### Corpus

In [67]:
beir_corpus_job_path = Path("../data/llm-gen/translated/beir_corpus_batch_result.jsonl")
download_job("batch_68180c78133881908cb8388e303a966e", beir_corpus_job_path)

Batch(
    id='batch_68180c78133881908cb8388e303a966e',
    completion_window='24h',
    created_at=1746406520,
    endpoint='/v1/chat/completions',
    input_file_id='file-G1fyzVU2mi2NXceLGPyQbQ',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1746409309,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746492920,
    failed_at=None,
    finalizing_at=1746409134,
    in_progress_at=1746406522,
    metadata=None,
    output_file_id='file-RPKHKA6RY4vWBuE2bzx5kE',
    request_counts=BatchRequestCounts(completed=1499, failed=0, total=1499)
)

In [68]:
df_corpus = pd.read_json("../data/cleaned/corpus.jsonl", lines=True).reset_index()
df_corpus.head()

,index,_id,title,text
0,0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,bismillah yuga lampah balukar janglar meunang ...
1,1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,KANYERI,eweuh deui seri nu lewih nyeri tibatan di héna...
2,2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,KARUNGING LAIN WAYAH,wanci gayuh ka peuting poék tungkeb haté lain ...
3,3,af9fc2c2-0230-4784-8104-cf9271ccfccc,KUDU DAÉK GAWÉ,ieu awak asa lalungsé di rasa asa carapé meure...
4,4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,PUISI PANGGGEUING DIRI,naha anjeun téh poho yén mot téh dodoho datang...


In [69]:
df_corpus_map = pd.read_json("../data/llm-gen/translated/beir_corpus_map.jsonl", lines=True)
df_corpus_map.head()

,custom_id,doc_index
0,41e2f3cd-4c56-40d3-876c-1cf0b83c35fb,0
1,134c29ff-604b-4c87-8b86-ac3ef807af04,1
2,1b1b0ab6-e6ac-4b66-94ab-98e2c89cabfa,2
3,10f8ced3-41ae-4bda-a435-067b84ab2ce0,3
4,5c415019-f192-415f-91e0-e97c756232b0,4


In [70]:
df_corpus_en = pd.DataFrame(gen_items(beir_corpus_job_path))
df_corpus_en.head()

FAILED CUSTOM ID: a71550fa-4d9c-4c16-8d8d-e9b578cacf24

,title,body,custom_id,total_tokens
0,WIDE PATIENCE,"In the name of Allah, may we act with the resu...",41e2f3cd-4c56-40d3-876c-1cf0b83c35fb,373
1,HURT,There is no other pain that is greater than th...,134c29ff-604b-4c87-8b86-ac3ef807af04,426
2,THE OTHER SIDE OF THE NIGHT,"as the evening approaches, heart tightly wrapp...",1b1b0ab6-e6ac-4b66-94ab-98e2c89cabfa,315
3,MUST BE WILLING TO WORK,"This body feels a sense of heaviness, as if bu...",10f8ced3-41ae-4bda-a435-067b84ab2ce0,304
4,POEM OF SELF REMEMBERING,"do you forget that death comes unannounced, no...",5c415019-f192-415f-91e0-e97c756232b0,388


In [71]:
print(f"Total tokens: {df_corpus_en['total_tokens'].sum()}")

Total tokens: 1104142

In [72]:
df_corpus_merged = df_corpus[["index", "_id"]].merge(df_corpus_map, left_on="index", right_on="doc_index")
df_corpus_merged = df_corpus_merged.merge(df_corpus_en, on="custom_id")
df_corpus_merged = df_corpus_merged.drop(columns=["index", "doc_index", "custom_id", "total_tokens"])
df_corpus_merged.head()

,_id,title,body
0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,WIDE PATIENCE,"In the name of Allah, may we act with the resu..."
1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,HURT,There is no other pain that is greater than th...
2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,THE OTHER SIDE OF THE NIGHT,"as the evening approaches, heart tightly wrapp..."
3,af9fc2c2-0230-4784-8104-cf9271ccfccc,MUST BE WILLING TO WORK,"This body feels a sense of heaviness, as if bu..."
4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,POEM OF SELF REMEMBERING,"do you forget that death comes unannounced, no..."


In [73]:
df_corpus_merged.to_json("../data/cleaned/corpus_en.jsonl", orient="records", lines=True)